#KnowledgeHub_RAG v0.4

## Objective
Improve retrieval quality using Hybrid Search.

### New Features
- BM25 keyword search
- FAISS semantic search
- Hybrid score fusion
- Better ranking quality

### Previous Version
v0.3 – Metadata-Aware Retrieval

In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install -q rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.3 MB/s eta 0:00:00


In [44]:
import os
import time
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from rank_bm25 import BM25Okapi

In [45]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 1 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf


In [49]:
def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        extracted = page.extract_text()

        if extracted:

            pages.append(
                {
                    "page": page_number,
                    "text": extracted
                }
            )

    return pages


documents = []

for pdf in pdf_files:

    pages = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "pages": pages
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 1 document(s).


In [50]:
def chunk_text(text, chunk_size=1000, overlap=200):

    paragraphs = text.split("\n\n")

    chunks = []

    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) <= chunk_size:

            current_chunk += paragraph + "\n\n"

        else:

            chunks.append(current_chunk.strip())

            current_chunk = current_chunk[-overlap:] + paragraph + "\n\n"

    if current_chunk:

        chunks.append(current_chunk.strip())

    return chunks

In [51]:
all_chunks = []

chunk_id = 0

for document in documents:

    for page in document["pages"]:

        chunks = chunk_text(page["text"])

        for chunk in chunks:

            all_chunks.append(
                {
                    "chunk_id": chunk_id,
                    "document": document["filename"],
                    "page": page["page"],
                    "text": chunk
                }
            )

            chunk_id += 1

print(f"Created {len(all_chunks)} chunks.")

Created 47 chunks.


In [53]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [54]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(47, 384)


In [55]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

47


In [56]:

tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print(" BM25 Index Built")

 BM25 Index Built


In [57]:
def retrieve(query, top_k=5,
             semantic_weight=0.6,
             keyword_weight=0.4):

    # ======================================
    # FAISS Retrieval
    # ======================================

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    faiss_scores, faiss_indices = index.search(
        query_embedding.reshape(1, -1),
        len(all_chunks)
    )

    faiss_scores = faiss_scores[0]
    faiss_indices = faiss_indices[0]

    # ======================================
    # BM25 Retrieval
    # ======================================

    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(query_tokens)

    # ======================================
    # Normalize Scores
    # ======================================

    faiss_norm = (
        faiss_scores - faiss_scores.min()
    ) / (
        faiss_scores.max() - faiss_scores.min() + 1e-8
    )

    bm25_norm = (
        bm25_scores - bm25_scores.min()
    ) / (
        bm25_scores.max() - bm25_scores.min() + 1e-8
    )

    # ======================================
    # Hybrid Score
    # ======================================

    results = []

    for rank, idx in enumerate(faiss_indices):

        semantic = float(faiss_norm[rank])

        keyword = float(bm25_norm[idx])

        hybrid = (
            semantic_weight * semantic +
            keyword_weight * keyword
        )

        results.append({

            "document": all_chunks[idx]["document"],

            "page": all_chunks[idx]["page"],

            "chunk_id": all_chunks[idx]["chunk_id"],

            "semantic_score": semantic,

            "bm25_score": keyword,

            "hybrid_score": hybrid,

            "text": all_chunks[idx]["text"]

        })

    results.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return results[:top_k]

In [58]:
results = retrieve(
    "What algorithm was used for classification?"
)

for r in results:

    print("=" * 90)

    print(f"Document        : {r['document']}")
    print(f"Page            : {r['page']}")
    print(f"Chunk ID        : {r['chunk_id']}")

    print()

    print(f"Semantic Score  : {r['semantic_score']:.4f}")
    print(f"BM25 Score      : {r['bm25_score']:.4f}")
    print(f"Hybrid Score    : {r['hybrid_score']:.4f}")

    print("-" * 90)

    print(r["text"][:500])

    print()

Document        : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page            : 8
Chunk ID        : 15

Semantic Score  : 1.0000
BM25 Score      : 1.0000
Hybrid Score    : 1.0000
------------------------------------------------------------------------------------------
3.4 Data Splitting and Validation
For each classifier, the available simulation dataset was partitioned into a training set (90%
of the data) and a validation set (10%) using stratified sampling to preserve the relative class
proportions. This ensures that each class including minority classes such as prompt collapse
events is adequately represented in both subsets.
To further avoid bias from an “unusually easy” or “unusually difficult” validation set, the
difficulty-aware splitting strategy des

Document        : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page            : 6
Chunk ID        : 11

Semantic Score  : 0.8362
BM25 Score      : 0.8778
Hybrid Score    : 0.8528
---------------

In [59]:
faiss.write_index(
    index,
    "knowledgehub.index"
)

In [60]:
loaded_index = faiss.read_index(
    "knowledgehub.index"
)

print(loaded_index.ntotal)

47


In [61]:
import time

start = time.time()

retrieve(
    "What algorithm was used for classification?"
)

print(
    f"Hybrid Retrieval Time: {time.time()-start:.4f} sec"
)

Hybrid Retrieval Time: 0.0701 sec


In [62]:
query = "Summarize the machine learning model."

results = retrieve(query)

print("=" * 90)
print("Hybrid Retrieval Results")
print("=" * 90)

for i, r in enumerate(results, 1):

    print()

    print(f"Source {i}")

    print(f"Document        : {r['document']}")
    print(f"Page            : {r['page']}")
    print(f"Chunk ID        : {r['chunk_id']}")

    print(f"Semantic Score  : {r['semantic_score']:.4f}")
    print(f"BM25 Score      : {r['bm25_score']:.4f}")
    print(f"Hybrid Score    : {r['hybrid_score']:.4f}")

    print("-" * 90)

Hybrid Retrieval Results

Source 1
Document        : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page            : 4
Chunk ID        : 7
Semantic Score  : 0.7888
BM25 Score      : 1.0000
Hybrid Score    : 0.8733
------------------------------------------------------------------------------------------

Source 2
Document        : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page            : 22
Chunk ID        : 42
Semantic Score  : 1.0000
BM25 Score      : 0.5545
Hybrid Score    : 0.8218
------------------------------------------------------------------------------------------

Source 3
Document        : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page            : 5
Chunk ID        : 9
Semantic Score  : 0.8249
BM25 Score      : 0.6053
Hybrid Score    : 0.7370
------------------------------------------------------------------------------------------

Source 4
Document        : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf